# World

> Run a policy in a pool of envs, to collect datasets or evaluate it (stable-worldmodel API).

In [ ]:
#| default_exp world.world

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

import contextlib
from collections import defaultdict
from functools import partial
from pathlib import Path
from typing import Any, Callable, Iterator

import gymnasium as gym
import numpy as np

from stable_marl.envs.registry import make
from stable_marl.policy import BasePolicy
from stable_marl.world.env_pool import EnvPool, episode_done
from stable_marl.wrappers.default import MegaWrapper

RESET_MODES = ('auto', 'wait')


def _make_env(env_name: str, wrappers: list, add_pixels: bool = False, **kwargs) -> gym.Env:
    if add_pixels:
        kwargs.setdefault('render_mode', 'rgb_array')
    env = make(env_name, **kwargs)
    for wrapper in wrappers:
        env = wrapper(env)
    return env

In [ ]:
#| export
class World:
    """
    Drives a policy through a pool of ``num_envs`` copies of a registered environment::

        world = World('MultiGrid-FindGoal-15x15-v0', num_envs=8, agents=2)
        world.set_policy(RandomPolicy(seed=0))
        world.collect('data/findgoal.h5', episodes=500, seed=0)    # record a dataset
        results = world.evaluate(episodes=100, seed=10_000)        # evaluate the policy

    Same interface as stable-worldmodel's ``World``: every env is wrapped with ``pre_wrappers``,
    then :class:`MegaWrapper` (everything into the info dict), then ``extra_wrappers``, and the
    pool stacks their infos. One difference in behaviour: episodes are assigned to the envs in
    advance (env ``i`` plays episodes ``i, i + num_envs, ...``) and episode ``k`` is reset with
    seed ``seed + k``, so results only depend on `seed` and the policy, not on `num_envs` or on
    which env finishes first.

    Attributes populated during a run:

    * ``infos``: stacked info dict of the last reset / step (see :class:`EnvPool`), plus
      ``_needs_flush`` (num_envs,): the envs that just started an episode
    * ``rewards``, ``terminateds``, ``truncateds``: (num_envs, num_agents) outputs of the last step

    Parameters
    ----------
    env_name : str
        Id of a registered environment (see ``stable_marl.list_envs()``)
    num_envs : int
        Number of envs in the pool
    image_shape : (H, W), optional
        Size the render is resized to (default: the render's size)
    max_episode_steps : int, optional
        Step limit of an episode (the env's ``max_steps``; default: the env's own)
    pre_wrappers : list of callables
        Applied to the raw env, before :class:`MegaWrapper` (e.g. ``FullyObsWrapper``)
    extra_wrappers : list of callables
        Applied after :class:`MegaWrapper`, on envs returning flat info dicts
    image_transform : callable, optional
        Applied to the render
    image_resample : str or int, optional
        PIL resampling used to resize the render (default bilinear)
    add_pixels : bool
        Add a render of the whole env to the infos, as ``pixels`` (as ``render`` if the env
        already provides ``pixels``, e.g. the agents' views of MultiGrid envs)
    goal_conditioned : bool
        Require a ``goal`` in the infos (envs with goals, e.g. FindGoal)
    goal_transform : callable, optional
        Applied to ``goal``
    **kwargs
        Passed to the env (e.g. ``agents=2``, ``tile_size=16``)
    """
    def __init__(
        self,
        env_name: str,
        num_envs: int = 1,
        image_shape: tuple[int, int] | None = None,
        max_episode_steps: int | None = None,
        pre_wrappers: list[Callable] | None = None,
        extra_wrappers: list[Callable] | None = None,
        image_transform: Callable | None = None,
        image_resample: str | int | None = None,
        add_pixels: bool = False,
        goal_conditioned: bool = False,
        goal_transform: Callable | None = None,
        **kwargs: Any):
        if max_episode_steps is not None:
            kwargs['max_steps'] = max_episode_steps
        wrappers = [
            *(pre_wrappers or []),
            partial(MegaWrapper, image_shape=image_shape, pixels_transform=image_transform,
                    image_resample=image_resample, add_pixels=add_pixels,
                    goal_transform=goal_transform, separate_goal=goal_conditioned),
            *(extra_wrappers or []),
        ]
        self.env_name, self.env_kwargs = env_name, kwargs
        self.envs = EnvPool([partial(_make_env, env_name, wrappers, add_pixels=add_pixels, **kwargs)] * num_envs)
        self.policy: BasePolicy | None = None
        self.infos: dict = {}
        self.rewards = self.terminateds = self.truncateds = None

    @property
    def num_envs(self) -> int:
        return self.envs.num_envs

    @property
    def num_agents(self) -> int:
        return self.envs.num_agents

    @property
    def metadata(self) -> dict[str, Any]:
        "Where a collected dataset comes from (stored with it)."
        return {'env_name': self.env_name, 'env_kwargs': self.env_kwargs, 'num_agents': self.num_agents}

    def close(self):
        self.envs.close()

    def set_policy(self, policy: BasePolicy):
        "Attach a policy, and apply its ``seed`` if it has one."
        self.policy = policy
        policy.set_env(self.envs)
        if hasattr(policy, '_set_seed') and getattr(policy, 'seed', None) is not None:
            policy._set_seed(policy.seed)

    def reset(self, seed=None, options=None):
        "Reset every env and refresh ``infos``."
        _, self.infos = self.envs.reset(seed=seed, options=options)
        self.infos['_needs_flush'] = np.ones(self.num_envs, bool)
        shape = (self.num_envs, self.num_agents)
        self.rewards, self.terminateds, self.truncateds = np.zeros(shape, np.float32), np.zeros(shape, bool), np.zeros(shape, bool)

In [ ]:
#| export
from fastcore.utils import patch

@patch
def _run_iter(
    self: World,
    episodes: int | None = None,
    seed: int | None = None,
    options: dict | None = None,
    mode: str = 'auto',
    on_step: Callable[[World, np.ndarray], None] | None = None) -> Iterator[tuple[int, int]]:
    """
    Play `episodes` episodes with the policy, yielding ``(env_index, episode_index)`` when one
    ends, before that env is reset (so its final state can still be read).

    `mode` 'auto' resets finished envs until `episodes` are played; 'wait' plays one episode per
    env (at most `episodes`) and leaves finished envs idle. ``on_step(world, mask)`` is called
    after every reset and step, `mask` marking the envs it concerns.
    """
    assert mode in RESET_MODES, f"reset_mode must be one of {RESET_MODES}"
    if self.policy is None:
        raise RuntimeError("No policy set: call `set_policy` first")
    E = self.num_envs
    episodes = E if episodes is None else episodes
    queues = [list(range(i, episodes, E))[:1 if mode == 'wait' else None] for i in range(E)]
    current = np.array([q.pop(0) if q else -1 for q in queues])
    active = current >= 0
    seed_of = lambda k: None if seed is None or k < 0 else seed + int(k)

    self.reset(seed=[seed_of(k) for k in current], options=options)
    if on_step: on_step(self, active)

    while active.any():
        actions = self.policy.get_action(self.infos)
        _, self.rewards, self.terminateds, self.truncateds, self.infos = self.envs.step(actions, mask=active)
        self.infos['_needs_flush'] = np.zeros(E, bool)
        if on_step: on_step(self, active)

        done = active & episode_done(self.terminateds, self.truncateds)
        for i in np.flatnonzero(done):
            yield int(i), int(current[i])
        if not done.any():
            continue

        restart = done & np.array([bool(q) for q in queues])
        current[done & ~restart], active[done & ~restart] = -1, False
        for i in np.flatnonzero(restart):
            current[i] = queues[i].pop(0)
        if restart.any():
            _, self.infos = self.envs.reset(seed=[seed_of(k) for k in current], options=options, mask=restart)
            self.terminateds[restart], self.truncateds[restart] = False, False
            self.infos['_needs_flush'] = restart.copy()
            if on_step: on_step(self, restart)

In [ ]:
#| export
@patch
def evaluate(
    self: World,
    episodes: int | None = None,
    seed: int | None = None,
    options: dict | None = None,
    video: str | Path | None = None,
    reset_mode: str | None = None,
    dataset: Any = None,
    episodes_idx: list[int] | None = None,
    start_steps: list[int] | None = None,
    goal_offset: int | None = None,
    eval_budget: int | None = None,
    callables: list[dict] | None = None) -> dict[str, Any]:
    """
    Run the policy and return, per episode (indexed by episode):

    * ``episode_successes``: whether the episode ended by termination (every agent terminated)
      rather than by the step limit
    * ``episode_returns`` (episodes, num_agents): sum of each agent's rewards
    * ``episode_lengths``: number of steps
    * ``seeds``: reset seed (-1 if unseeded)

    and ``success_rate`` (in percent, as in stable-worldmodel), ``mean_return`` (summed over
    agents) and ``mean_length``.

    Two modes, as in stable-worldmodel:

    * **episodic** (default): `episodes` episodes; `reset_mode` 'wait' plays one episode per env
    * **dataset-driven**: with `dataset`, env ``i`` starts from step ``start_steps[i]`` (default 0)
      of episode ``episodes_idx[i]`` (``num_envs`` episodes) and must reach the state recorded
      `goal_offset` steps later within `eval_budget` steps. Envs with a
      ``reset_options_from_dataset(init_row, goal_row)`` method (MultiGrid envs) are reset with
      the options it returns; for the others, the ``callables`` specs
      (``{'method': name, 'args': {arg: {'value': ..., 'in_dataset': bool}}}``) are called on the
      unwrapped env after a reset, with a dataset column as value if ``in_dataset``. The goal
      (``goal`` from the recorded `pixels`, ``goal_<col>`` from the other columns) is written
      into the infos at every step.

    `video`: a directory for one mp4 per episode (episodic: ``episode_<k>.mp4``; dataset-driven:
    ``env_<i>.mp4``, the run next to the recorded steps and the goal), made from the infos'
    ``render`` (the env render), or else ``pixels`` (the agents' views side by side).
    """
    if dataset is not None:
        return self._evaluate_from_dataset(dataset, episodes_idx, start_steps, goal_offset, eval_budget,
                                           callables, video, reset_mode or 'wait')
    mode = reset_mode or 'auto'
    E, A = self.num_envs, self.num_agents
    n = E if episodes is None else (episodes if mode == 'auto' else min(episodes, E))
    results = _empty_results(n, A)
    returns = np.zeros((E, A))
    frames: dict[int, list] | None = defaultdict(list) if video else None

    def on_step(world, mask):
        for i in np.flatnonzero(mask):
            new = world.infos['_needs_flush'][i]
            returns[i] = 0 if new else returns[i] + world.infos['reward'][i, 0]
            if frames is not None:
                if new: frames[i] = []
                frames[i].append(_video_frame(world.infos, i))

    for i, k in self._run_iter(n, seed=seed, options=options, mode=mode, on_step=on_step):
        results['episode_successes'][k] = self.terminateds[i].all()
        results['episode_returns'][k] = returns[i]
        results['episode_lengths'][k] = self.infos['step_idx'][i, 0]
        results['seeds'][k] = self.envs.seeds[i]
        if frames is not None:
            from stable_marl.plot.video_utils import save_video
            save_video(Path(video) / f'episode_{k}.mp4', frames.pop(i))
    return _summarize(results)


def _empty_results(n: int, num_agents: int) -> dict[str, Any]:
    return {
        'success_rate': 0.0,
        'episode_successes': np.zeros(n, bool),
        'episode_returns': np.zeros((n, num_agents)),
        'episode_lengths': np.zeros(n, np.int64),
        'seeds': np.full(n, -1, np.int64),
    }


def _summarize(results: dict[str, Any]) -> dict[str, Any]:
    if len(results['episode_successes']):
        results['success_rate'] = float(results['episode_successes'].mean()) * 100.0
        results['mean_return'] = float(results['episode_returns'].sum(axis=1).mean())
        results['mean_length'] = float(results['episode_lengths'].mean())
    return results


def _video_key(infos: dict) -> str:
    for key in ('render', 'pixels'):
        if key in infos:
            return key
    raise ValueError("evaluate(video=...) needs images in the infos: `pixels` or, with add_pixels=True, `render`")


def _side_by_side(image: np.ndarray) -> np.ndarray:
    "An image (H, W, C), or per-agent images (num_agents, H, W, C) side by side."
    image = np.asarray(image)
    return np.concatenate(list(image), axis=1) if image.ndim == 4 else image


def _video_frame(infos: dict, i: int) -> np.ndarray:
    return _side_by_side(infos[_video_key(infos)][i, 0]).copy()


In [ ]:
#| export
@patch
def _evaluate_from_dataset(
    self: World, dataset, episodes_idx, start_steps, goal_offset, eval_budget, callables, video, mode) -> dict:
    from copy import deepcopy
    n, A = self.num_envs, self.num_agents
    if episodes_idx is None or len(episodes_idx) != n:
        raise ValueError(f"dataset-driven evaluation needs one episode per env: {n} `episodes_idx`")
    if goal_offset is None or eval_budget is None:
        raise ValueError("dataset-driven evaluation needs a `goal_offset` and an `eval_budget`")
    if mode != 'wait':
        raise ValueError("dataset-driven evaluation plays one run per env: reset_mode must be 'wait'")
    if self.policy is None:
        raise RuntimeError("No policy set: call `set_policy` first")
    start_steps = [0] * n if start_steps is None else list(start_steps)
    infos_key = 'render' if 'render' in (dataset.column_names or []) and video else 'pixels'
    init_rows, goal_rows, dataset_videos = _extract_init_goal(dataset, episodes_idx, start_steps, goal_offset, infos_key)

    seeds = None
    if 'seed' in init_rows[0]:
        seeds = [int(np.asarray(row['seed']).reshape(-1)[0]) for row in init_rows]
        seeds = [s if s >= 0 else None for s in seeds]
    unwrapped = [env.unwrapped for env in self.envs.envs]
    restores = hasattr(unwrapped[0], 'reset_options_from_dataset')
    if restores:
        self.reset(seed=seeds, options=[u.reset_options_from_dataset(init_rows[i], goal_rows[i])
                                        for i, u in enumerate(unwrapped)])
    else:
        self.reset(seed=seeds)
        for i in range(n):
            _apply_callables(unwrapped[i], callables or [], {**init_rows[i], **goal_rows[i]})

    # dataset values into the infos (one row per env, shape (num_envs, 1, ...)): the goal always; the
    # start step only when the env was set up by `callables` (restored envs report their own state)
    episode_cols = set(getattr(dataset, 'episode_column_names', None) or [])
    goal_keys = set(goal_rows[0])
    for rows in ((init_rows, goal_rows) if not restores else (goal_rows,)):
        for key in rows[0]:
            if key in episode_cols or (key not in self.infos and key not in goal_keys):
                continue
            vals = [row[key] for row in rows]
            if not all(isinstance(v, np.ndarray) for v in vals) or len({v.shape for v in vals}) > 1:
                continue
            self.infos[key] = np.stack(vals)[:, None].copy()
    goal_snapshot = {k: self.infos[k].copy() for k in goal_keys if k in self.infos}

    results = _empty_results(n, A)
    results['seeds'] = np.array([-1 if s is None else s for s in seeds] if seeds else [-1] * n, np.int64)
    frames = [[_video_frame(self.infos, i)] for i in range(n)] if video else None
    alive = np.ones(n, bool)
    for _ in range(eval_budget):
        actions = self.policy.get_action(self.infos)
        _, self.rewards, self.terminateds, self.truncateds, self.infos = self.envs.step(actions, mask=alive)
        self.infos['_needs_flush'] = np.zeros(n, bool)
        self.infos.update(deepcopy(goal_snapshot))
        results['episode_returns'][alive] += self.rewards[alive]
        results['episode_lengths'][alive] += 1
        results['episode_successes'] |= alive & self.terminateds.all(axis=1)
        if frames is not None:
            for i in np.flatnonzero(alive):
                frames[i].append(_video_frame(self.infos, i))
        alive &= ~episode_done(self.terminateds, self.truncateds)
        if not alive.any():
            break

    if frames is not None:
        from stable_marl.plot.video_utils import save_panel_videos
        save_panel_videos(Path(video), {'agent': frames, 'dataset': dataset_videos,
                                        'goal': [_side_by_side(row['goal']) for row in goal_rows]})
    return _summarize(results)


def _as_numpy_image(col: str, arr: np.ndarray) -> np.ndarray:
    "Undo the channel-first layout datasets give single images (T, H, W, C) (see `to_tensor`)."
    if arr.ndim == 4 and arr.shape[1] in (1, 3) and arr.shape[-1] not in (1, 3):
        return arr.transpose(0, 2, 3, 1)
    return arr


def _extract_init_goal(dataset, episodes_idx, start_steps, goal_offset, video_key: str = 'pixels'):
    """
    Per-episode rows of a dataset-driven evaluation: ``init_rows[i]`` (every column at the start
    step, plus the episode columns), ``goal_rows[i]`` (the step `goal_offset` later, as ``goal``
    for ``pixels`` and ``goal_<col>`` for the others) and ``dataset_videos[i]`` (the recorded
    `video_key` images in between).
    """
    ep_idx, start = np.asarray(episodes_idx), np.asarray(start_steps)
    if getattr(dataset, 'lengths', None) is not None:
        lengths = np.asarray(dataset.lengths)[ep_idx]
        if (start < 0).any() or (start + goal_offset >= lengths).any():
            raise ValueError(f"start_steps + goal_offset must stay within the episodes (lengths {lengths.tolist()})")
    data = dataset.load_chunk(ep_idx, start, start + goal_offset + 1)
    episode_cols = list(getattr(dataset, 'episode_column_names', None) or [])
    episode_data = dataset.get_episode_data(list(episodes_idx)) if episode_cols else {}
    init_rows, goal_rows, videos = [], [], []
    for i, ep in enumerate(data):
        init_row, goal_row = {}, {}
        for col, val in ep.items():
            if col.startswith('goal') or col == 'action' or not hasattr(val, 'shape'):
                continue
            arr = _as_numpy_image(col, val.numpy() if hasattr(val, 'numpy') else np.asarray(val))
            init_row[col] = arr[0]
            goal_row['goal' if col == 'pixels' else f'goal_{col}'] = arr[-1]
            if col == video_key:
                videos.append(arr)
        for col in episode_cols:
            init_row[col] = episode_data[col][i]
        init_rows.append(init_row)
        goal_rows.append(goal_row)
    return init_rows, goal_rows, videos


def _apply_callables(env, callables: list[dict], init_state: dict):
    "Call ``env.<method>(**args)`` for each spec, with dataset values for the args marked ``in_dataset``."
    from copy import deepcopy
    for spec in callables:
        method = spec['method']
        if not hasattr(env, method):
            continue
        prepared = {}
        for name, data in spec.get('args', {}).items():
            if data.get('in_dataset', True):
                if data.get('value') in init_state:
                    prepared[name] = deepcopy(init_state[data['value']])
            else:
                prepared[name] = data.get('value')
        getattr(env, method)(**prepared)

In [ ]:
#| export
@patch
def collect(
    self: World,
    path: str | Path | None = None,
    episodes: int = 0,
    seed: int | None = None,
    options: dict | None = None,
    format: str = 'hdf5',
    writer: Any = None,
    progress: bool = True):
    """
    Play `episodes` episodes and write them to `path` in the dataset `format` (or to a
    ready-made `writer`, e.g. a writer opened with ``mode='overwrite'``).

    Every info key becomes a column with one row per step, from the reset to the last step,
    as in stable-worldmodel: row ``t`` holds the infos at step ``t``, whose ``reward``,
    ``terminated`` and ``truncated`` come from the step that led there (``nan`` / False on
    the first row), while ``action`` is the action taken at step ``t`` (``nan`` on the last row).
    Keys starting with ``_`` and non-array values are skipped.

    Envs with a ``get_episode_data()`` method (on the unwrapped env: values constant within an
    episode, e.g. a layout description) have it attached to each episode under
    ``EPISODE_DATA_KEY``, for writers that store it (e.g. a :class:`ReplayBuffer`).
    """
    from stable_marl.data import EPISODE_DATA_KEY, get_format

    if (path is None) == (writer is None):
        raise ValueError("World.collect: pass exactly one of `path` or `writer`")
    writer = get_format(format).open_writer(path) if writer is None else writer
    if hasattr(writer, 'metadata'):
        writer.metadata.update(self.metadata)
    buffers = [defaultdict(list) for _ in range(self.num_envs)]

    def on_step(world, mask):
        for col, data in world.infos.items():
            if col.startswith('_') or not isinstance(data, np.ndarray):
                continue
            for i in np.flatnonzero(mask):
                buffers[i][col].append(data[i, 0].copy())

    bar = _progress_bar(episodes, 'Recording') if progress else contextlib.nullcontext()
    with writer as w, bar as pbar:
        def episode_iter():
            for i, _ in self._run_iter(episodes, seed=seed, options=options, mode='auto', on_step=on_step):
                episode = {col: np.stack(rows) for col, rows in buffers[i].items()}
                buffers[i].clear()
                if 'action' in episode:   # action taken at each step: the reset's nan moves last
                    episode['action'] = np.roll(episode['action'], -1, axis=0)
                # yielded before the env's reset: it still holds the episode that just ended
                ep_data_fn = getattr(self.envs.envs[i].unwrapped, 'get_episode_data', None)
                extra = ep_data_fn() if ep_data_fn is not None else None
                if extra:
                    episode[EPISODE_DATA_KEY] = dict(extra)
                if pbar is not None: pbar.update(1)
                yield episode
        w.write_episodes(episode_iter())


def _progress_bar(total: int, desc: str):
    try:
        from tqdm import tqdm
    except ImportError:
        return contextlib.nullcontext()
    return tqdm(total=total, desc=desc)

### Tests

In [ ]:
import tempfile, torch
from stable_marl.policy import RandomPolicy, FeedForwardPolicy
from stable_marl.envs.multigrid import GoToGoalPolicy
from stable_marl.data import HDF5Dataset
from stable_marl.wrappers.base import FullyObsWrapper

world = World('MultiGrid-Empty-6x6-v0', num_envs=3, agents=2)
world.set_policy(RandomPolicy(seed=0))
res = world.evaluate(episodes=7, seed=100)
assert list(res['seeds']) == list(range(100, 107)) and res['episode_returns'].shape == (7, 2)
assert len(world.evaluate(seed=0, reset_mode='wait')['seeds']) == 3

# the expert reaches the goal, and results do not depend on the number of envs
for num_envs in (1, 3):
    world = World('MultiGrid-Empty-6x6-v0', num_envs=num_envs, agents=2)
    world.set_policy(GoToGoalPolicy())
    res = world.evaluate(episodes=5, seed=0)
    assert res['success_rate'] == 100.0, res
    if num_envs == 1: ref = res
    else: assert np.array_equal(ref['episode_lengths'], res['episode_lengths'])

# a torch model as policy
class Turn(torch.nn.Module):
    def get_action(self, info):
        return torch.zeros(info['direction'].shape[0], info['direction'].shape[2], dtype=torch.long)
world.set_policy(FeedForwardPolicy(Turn()))
assert world.evaluate(episodes=2, seed=0)['success_rate'] == 0.0

# the render resized, and a pre-wrapper changing the observations
world = World('MultiGrid-Empty-6x6-v0', num_envs=2, agents=2, add_pixels=True, image_shape=(32, 32),
              pre_wrappers=[FullyObsWrapper])
world.set_policy(RandomPolicy(seed=0))
world.evaluate(episodes=2, seed=0)
assert world.infos['render'].shape == (2, 1, 32, 32, 3) and world.infos['image'].shape == (2, 1, 2, 6, 6, 3)

# collect -> hdf5 dataset
with tempfile.TemporaryDirectory() as tmp:
    world = World('MultiGrid-Empty-6x6-v0', num_envs=2, agents=2, max_episode_steps=10)
    world.set_policy(RandomPolicy(seed=1))
    world.collect(f'{tmp}/empty.h5', episodes=3, seed=5, progress=False)
    ds = HDF5Dataset(path=f'{tmp}/empty.h5', num_steps=4)
    assert len(ds.lengths) == 3 and ds.metadata['env_name'] == 'MultiGrid-Empty-6x6-v0'
    item = ds[0]
    assert item['pixels'].shape == (4, 2, 224, 224, 3) and item['action'].shape == (4, 2)
    assert {'image', 'state', 'reward', 'terminated', 'truncated', 'step_idx', 'id', 'seed'} <= set(ds.column_names)

# goals and variations end up in the dataset
with tempfile.TemporaryDirectory() as tmp:
    world = World('MultiGrid-FindGoal-15x15-v0', num_envs=2, agents=2, num_obstacles=6, goal_conditioned=True)
    world.set_policy(GoToGoalPolicy())
    world.collect(f'{tmp}/fg.h5', episodes=4, seed=0, options={'variation': ['wall.color', 'agent.color']}, progress=False)
    ds = HDF5Dataset(path=f'{tmp}/fg.h5')
    assert {'goal', 'goal_position', 'variation.wall.color', 'variation.agent.color'} <= set(ds.column_names)
    colors = ds.get_col_data('variation.wall.color')[ds.offsets]
    assert len(set(colors.tolist())) > 1   # one sampled wall color per episode
# a video per episode: the render (or the agents' views side by side)
from pathlib import Path
from fastcore.test import test_fail
with tempfile.TemporaryDirectory() as tmp:
    world = World('MultiGrid-Empty-6x6-v0', num_envs=2, agents=2, add_pixels=True, image_shape=(32, 32), max_episode_steps=6)
    world.set_policy(RandomPolicy(seed=0))
    world.evaluate(episodes=3, seed=0, video=f'{tmp}/render')
    assert sorted(p.name for p in Path(f'{tmp}/render').iterdir()) == ['episode_0.mp4', 'episode_1.mp4', 'episode_2.mp4']
    world = World('MultiGrid-Empty-6x6-v0', num_envs=1, agents=2, max_episode_steps=4, tile_size=8)
    world.set_policy(RandomPolicy(seed=0))
    world.evaluate(episodes=1, seed=0, video=f'{tmp}/views')
    import imageio
    assert imageio.v3.improps(f'{tmp}/views/episode_0.mp4').shape[1:3] == (64, 112)   # two 56 x 56 views (height padded to 16)

# dataset-driven evaluation: each env starts from a recorded step and must reach the step goal_offset later
from stable_marl.data import HDF5Dataset
with tempfile.TemporaryDirectory() as tmp:
    kw = dict(agents=2, size=9, num_obstacles=4, n_clutter=0, min_goal_spawn_distance=3, tile_size=8, max_steps=40)
    world = World('MultiGrid-FindGoal-15x15-v0', num_envs=2, goal_conditioned=True, **kw)
    world.set_policy(GoToGoalPolicy())
    world.collect(f'{tmp}/fg.h5', episodes=4, seed=3, options={'variation': ['wall.color']}, progress=False)
    ds = HDF5Dataset(path=f'{tmp}/fg.h5')
    eps, starts = [1, 2], [2, 1]

    # the restored start: same layout (seed, recorded wall color), agents and their targets
    res = world.evaluate(dataset=ds, episodes_idx=eps, start_steps=starts, goal_offset=1, eval_budget=0)
    for i, (e, s) in enumerate(zip(eps, starts)):
        row = ds.get_row_data(int(ds.offsets[e]) + s)
        assert np.array_equal(world.infos['state'][i, 0], row['state']), i
        assert np.array_equal(world.infos['position'][i, 0], row['position'])
        nxt = ds.get_row_data(int(ds.offsets[e]) + s + 1)
        assert np.array_equal(world.infos['goal_position'][i, 0], nxt['position'])        # the goal: one step later
        assert np.array_equal(world.infos['goal'][i, 0], nxt['pixels'])
        assert np.array_equal(world.envs.envs[i].unwrapped._targets, nxt['position'])
    # episodes are stored in the order they ended: their reset seeds come from the dataset
    assert list(res['seeds']) == list(ds.get_col_data('seed')[ds.offsets[eps]]) and list(res['episode_lengths']) == [0, 0]

    # the expert replays the recorded step: every agent is on its target after one step
    res = world.evaluate(dataset=ds, episodes_idx=eps, start_steps=starts, goal_offset=1, eval_budget=5)
    assert res['success_rate'] == 100.0 and list(res['episode_lengths']) == [1, 1], res
    # ... and reaches the end of the episodes, with the goal kept in the infos at every step
    ends = [int(ds.lengths[e]) - 1 - s for e, s in zip(eps, starts)]
    res = world.evaluate(dataset=ds, episodes_idx=eps, start_steps=starts, goal_offset=min(ends), eval_budget=30, video=f'{tmp}/ds')
    assert res['success_rate'] == 100.0, res
    assert sorted(p.name for p in Path(f'{tmp}/ds').iterdir()) == ['env_0.mp4', 'env_1.mp4']
    # a policy that only turns never gets there (unless an agent's target is its start cell)
    world.set_policy(FeedForwardPolicy(Turn()))
    res = world.evaluate(dataset=ds, episodes_idx=eps, start_steps=[0, 0], goal_offset=3, eval_budget=6)
    assert res['success_rate'] == 0.0 and list(res['episode_lengths']) == [6, 6]

    test_fail(lambda: world.evaluate(dataset=ds, episodes_idx=[0], goal_offset=1, eval_budget=3), contains='one episode per env')
    test_fail(lambda: world.evaluate(dataset=ds, episodes_idx=eps, eval_budget=3), contains='goal_offset')
    test_fail(lambda: world.evaluate(dataset=ds, episodes_idx=eps, goal_offset=1, eval_budget=3, reset_mode='auto'), contains="'wait'")
    test_fail(lambda: world.evaluate(dataset=ds, episodes_idx=eps, goal_offset=500, eval_budget=3), contains='within the episodes')
    # restoring needs the reset seeds
    world.set_policy(GoToGoalPolicy())
    world.collect(f'{tmp}/unseeded.h5', episodes=2, progress=False)
    test_fail(lambda: world.evaluate(dataset=HDF5Dataset(path=f'{tmp}/unseeded.h5'), episodes_idx=[0, 1], goal_offset=1,
                                     eval_budget=3), contains='seed')

# envs without reset_options_from_dataset: `callables` set them up from the dataset rows
class _Env:
    def set_state(self, pos, speed): self.got = (pos, speed)
env = _Env()
_apply_callables(env, [{'method': 'set_state', 'args': {'pos': {'value': 'position', 'in_dataset': True},
                                                       'speed': {'value': 3, 'in_dataset': False}}},
                       {'method': 'missing', 'args': {}}], {'position': np.array([1, 2])})
assert env.got[1] == 3 and list(env.got[0]) == [1, 2]


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()